In [27]:
# ============================================================
# Task 1 forecasting model is recreated below and used
# to estimate gas prices required for contract valuation.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# ------------------------------------------------------------
# 1. LOAD AND PREPARE DATA
# ------------------------------------------------------------

file_path = "Nat_Gas(1).csv"   # ensure file is in same directory
df = pd.read_csv(file_path)

# Convert date column and clean
df["Dates"] = pd.to_datetime(
    df["Dates"],
    format="%m/%d/%y"
)
df = df.sort_values("Dates")
df.set_index("Dates", inplace=True)

# Rename price column for clarity
df.rename(columns={"Prices": "Price"}, inplace=True)

# ------------------------------------------------------------
# 2. FEATURE ENGINEERING
# ------------------------------------------------------------

# Create time index for trend modeling
df["time_index"] = np.arange(len(df))

# Extract month for seasonality
df["month"] = df.index.month

# Create monthly dummy variables (seasonal effects)
month_dummies = pd.get_dummies(df["month"], prefix="month", drop_first=True)

# Define independent (X) and dependent (y) variables
X = pd.concat([df["time_index"], month_dummies], axis=1)
y = df["Price"]

# ------------------------------------------------------------
# 3. MODEL FITTING (Trend + Seasonality)
# ------------------------------------------------------------

model = LinearRegression()
model.fit(X, y)

# Generate fitted historical values
df["fitted_price"] = model.predict(X)

# ------------------------------------------------------------
# 4. FORECAST NEXT 12 MONTHS
# ------------------------------------------------------------

future_dates = pd.date_range(
    start=df.index[-1] + pd.offsets.MonthEnd(),
    periods=12,
    freq="ME"
)

future_df = pd.DataFrame(index=future_dates)

# Create future features
future_df["time_index"] = np.arange(len(df), len(df) + 12)
future_df["month"] = future_df.index.month

future_month_dummies = pd.get_dummies(future_df["month"], prefix="month")

# Align columns with training data
future_month_dummies = future_month_dummies.reindex(
    columns=month_dummies.columns,
    fill_value=0
)

X_future = pd.concat([future_df["time_index"], future_month_dummies], axis=1)

# Predict future prices
future_df["predicted_price"] = model.predict(X_future)

# ------------------------------------------------------------
# 5. CREATE DAILY INTERPOLATED SERIES
# ------------------------------------------------------------

# Combine historical fitted and future predicted prices
monthly_series = pd.concat([
    df["fitted_price"],
    future_df["predicted_price"]
])

# Create full daily range
daily_index = pd.date_range(
    start=df.index[0],
    end=future_df.index[-1],
    freq="D"
)

daily_df = pd.DataFrame(index=daily_index)
daily_df["price"] = monthly_series.reindex(daily_df.index)

# Use linear interpolation for smooth daily estimates
daily_df["price"] = daily_df["price"].interpolate(method="linear")

# ------------------------------------------------------------
# 6. PRICE QUERY FUNCTION
# ------------------------------------------------------------

def get_gas_price(input_date):
    """
    Returns estimated natural gas price for a given date.
    
    Parameters:
        input_date (str or datetime): Date to query
        
    Returns:
        float: Estimated gas price
    """
    input_date = pd.to_datetime(input_date)

    if input_date < daily_df.index[0] or input_date > daily_df.index[-1]:
        return "Date outside model range"

    return round(float(daily_df.loc[input_date]["price"]),2)

# ============================================================
# Natural Gas Contract Valuation
# Task 2 code function begins here
# ============================================================

def price_storage_contract(
    injections,
    withdrawals,
    max_volume,
    injection_rate,
    withdrawal_rate,
    storage_cost_per_month
):
    """
    Calculates the value of a natural gas storage contract.

    Parameters
    ----------
    injections : list of tuples
        List containing (date, volume) injection events.

    withdrawals : list of tuples
        List containing (date, volume) withdrawal events.

    max_volume : float
        Maximum storage capacity.

    injection_rate : float
        Maximum amount of gas that can be injected per transaction.

    withdrawal_rate : float
        Maximum amount of gas that can be withdrawn per transaction.

    storage_cost_per_month : float
        Fixed monthly storage fee charged by the storage facility.

    Returns
    -------
    float
        Estimated contract value.
    """

    # Current inventory level in storage
    inventory = 0

    # Track total costs and revenues
    purchase_cost = 0
    sales_revenue = 0
    storage_cost = 0

    # Combine all injection and withdrawal events
    events = []

    # Add injection events
    for date, volume in injections:
        events.append((pd.to_datetime(date), "inject", volume))

    # Add withdrawal events
    for date, volume in withdrawals:
        events.append((pd.to_datetime(date), "withdraw", volume))

    # Sort events chronologically
    events.sort(key=lambda x: x[0])

    # First event date used for storage cost calculation
    last_date = events[0][0]

    # Process each event in order
    for date, action, volume in events:

        # Calculate time elapsed since previous event
        months_held = (date - last_date).days / 30

        # Storage cost is based on inventory held during that period
        storage_cost += (
            storage_cost_per_month *
            months_held
        )

        # -----------------------------
        # INJECTION EVENT
        # -----------------------------
        if action == "inject":

            # Check injection rate limit
            if volume > injection_rate:
                raise ValueError(
                    f"Injection rate exceeded on {date.date()}"
                )

            # Check storage capacity
            if inventory + volume > max_volume:
                raise ValueError(
                    f"Storage capacity exceeded on {date.date()}"
                )

            # Obtain market price from Task 1 model
            price = get_gas_price(date)

            # Cost of purchasing gas
            purchase_cost += volume * price

            # Update inventory
            inventory += volume

        # -----------------------------
        # WITHDRAWAL EVENT
        # -----------------------------
        else:

            # Check withdrawal rate limit
            if volume > withdrawal_rate:
                raise ValueError(
                    f"Withdrawal rate exceeded on {date.date()}"
                )

            # Ensure enough gas is available
            if volume > inventory:
                raise ValueError(
                    f"Insufficient inventory on {date.date()}"
                )

            # Obtain market price from Task 1 model
            price = get_gas_price(date)

            # Revenue from selling gas
            sales_revenue += volume * price

            # Update inventory
            inventory -= volume

        # Move to next event
        last_date = date

    # Contract value calculation
    contract_value = (
        sales_revenue
        - purchase_cost
        - storage_cost
    )

    return round(contract_value, 2)

In [29]:
value = price_storage_contract(
    injections=[
        ("2024-06-01", 100000),
        ("2024-07-01", 50000)
    ],
    withdrawals=[
        ("2024-11-01", 75000),
        ("2024-12-01", 75000)
    ],
    max_volume=200000,
    injection_rate=100000,
    withdrawal_rate=100000,
    storage_cost_per_month=10000
)

print(f"Contract Value: ${value:,.2f}")

Contract Value: $64,750.00
